In [1]:
#데이터 및 모델, 라이브러리
import pandas as pd
import joblib
from scipy.sparse import csr_matrix, hstack

df = pd.read_csv("cve_preprocessed.csv")
model = joblib.load("risk_model.pkl")

In [2]:
label_encoder = joblib.load("feature_data.pkl")["label_encoder"]
tfidf_vectorizer = joblib.load("tfidf_vectorizer.pkl")
feature_cols = joblib.load("feature_cols.pkl")
encoded_columns = joblib.load("encoded_columns.pkl")

# 1. 범주형 원-핫 인코딩 (학습 때와 동일 방식)
encoded_features = pd.get_dummies(
    df[feature_cols],
    prefix=feature_cols,
    dtype=int,
)
encoded_features = encoded_features.reindex(columns=encoded_columns, fill_value=0)

# 2. description TF-IDF 변환 (transform만! fit 하면 안 됨)
description_features = tfidf_vectorizer.transform(df["description"])

# 3. 두 결과 결합 (학습 때와 같은 순서)
categorical_features = csr_matrix(encoded_features.values)
X_new = hstack([categorical_features, description_features], format="csr")

# 4. 예측 + 라벨 복원
df["predicted_severity"] = label_encoder.inverse_transform(model.predict(X_new))

print("예측 완료")
print(df[["cve_id", "predicted_severity"]].head())

예측 완료
           cve_id predicted_severity
0  CVE-2026-57752               HIGH
1  CVE-2026-57753               HIGH
2  CVE-2026-57754             MEDIUM
3  CVE-2026-57755             MEDIUM
4  CVE-2026-57756               HIGH


In [3]:
#점수표
severity_score_map = {
    "LOW": 25,
    "MEDIUM": 50,
    "HIGH": 75,
    "CRITICAL": 100,
}

attack_vector_score_map = {
    "PHYSICAL": 20,
    "LOCAL": 40,
    "ADJACENT_NETWORK": 70,
    "NETWORK": 100,
}

attack_complexity_score_map = {
    "HIGH": 40,
    "LOW": 100,
}

privileges_score_map = {
    "HIGH": 30,
    "LOW": 70,
    "NONE": 100,
}

user_interaction_score_map = {
    "REQUIRED": 40,
    "NONE": 100,
}

In [4]:
#우선순위 함수
def calculate_priority_score(row) -> float:
    cvss_score = float(row["cvss_score"]) * 10

    severity_score = severity_score_map.get(
        row["predicted_severity"],
        0,
    )

    vector_score = attack_vector_score_map.get(
        row["attack_vector"],
        0,
    )

    complexity_score = attack_complexity_score_map.get(
        row["attack_complexity"],
        0,
    )

    privilege_score = privileges_score_map.get(
        row["privileges_required"],
        0,
    )

    interaction_score = user_interaction_score_map.get(
        row["user_interaction"],
        0,
    )

    score = (
        cvss_score * 0.50
        + severity_score * 0.20
        + vector_score * 0.10
        + complexity_score * 0.08
        + privilege_score * 0.07
        + interaction_score * 0.05
    )

    return round(score, 2)

In [5]:
#점수와 순위 생성
df["priority_score"] = df.apply(
    calculate_priority_score,
    axis=1,
)

df = df.sort_values(
    by="priority_score",
    ascending=False,
).reset_index(drop=True)

df["priority_rank"] = df.index + 1

display(
    df[
        [
            "priority_rank",
            "cve_id",
            "cvss_score",
            "predicted_severity",
            "attack_vector",
            "attack_complexity",
            "privileges_required",
            "user_interaction",
            "priority_score",
        ]
    ].head(20)
)

#데이터 분포 확인
print(df["priority_score"].value_counts().head(10))

,priority_rank,cve_id,cvss_score,predicted_severity,attack_vector,attack_complexity,privileges_required,user_interaction,priority_score
0,1,CVE-2026-56004,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
1,2,CVE-2026-50746,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
2,3,CVE-2026-43898,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
3,4,CVE-2026-35307,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
4,5,CVE-2026-13768,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
5,6,CVE-2026-47140,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
6,7,CVE-2026-47208,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
7,8,CVE-2026-35308,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
8,9,CVE-2026-44329,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
9,10,CVE-2026-46846,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0


priority_score
82.5    796
59.4    734
99.0    425
86.0    406
69.5    362
75.9    333
70.4    327
58.5    324
77.5    278
86.9    274
Name: count, dtype: int64


In [6]:
print("실제 정답 분포")
print(df["severity"].value_counts())

print("\n모델 예측 분포")
print(df["predicted_severity"].value_counts())

실제 정답 분포
severity
HIGH        4352
MEDIUM      4101
CRITICAL    1194
LOW          347
Name: count, dtype: int64

모델 예측 분포
predicted_severity
MEDIUM      4293
HIGH        4086
CRITICAL    1238
LOW          377
Name: count, dtype: int64


In [7]:
#csv 생성
df.to_csv(
    "priority.csv",
    index=False,
    encoding="utf-8-sig"
)

print("priority.csv 생성 완료")
print("결과 크기:", df.shape)

priority.csv 생성 완료
결과 크기: (9994, 12)
